# 03 - Descriptive Network Analysis

This notebook takes the trip-adjacency graph built in notebook `02_graph_construction` and answers the first question any network study has to answer before it can talk about resilience: **what does this network actually look like?** We measure size (nodes, edges), sparsity (density, average degree), fragmentation (connected components, largest-component share), the shape of the degree distribution (linear histogram plus a log-log view to check for a heavy tail), and the two classical local-connectivity structures - **articulation points** (vertices whose removal disconnects the graph) and **bridges** (edges whose removal disconnects the graph). Those last two are the direct link to the project's research question, because they are the places where the network has no redundancy at all.

**Research question addressed here:** how sparse and how fragile is the Israeli public-transport network at the structural level, and how many single stations / single segments are there whose loss would split the network?

## Inputs

* `outputs/nb/02_graph_construction/nodes.csv` - one row per active stop with `stop_id, stop_name, lat, lon, region, metro`.
* `outputs/nb/02_graph_construction/edges.csv` - one row per **directed** segment `from_stop, to_stop, trip_frequency`.

Both are produced by notebook `02_graph_construction`. This notebook does **not** read the raw GTFS feed and does **not** need the 816 MB `stop_times.txt` file.

## Outputs

Everything is written under `outputs/nb/03_descriptive_analysis/`:

* `network_summary.json` - all headline statistics in one dictionary.
* `tables/network_summary.csv` - the same statistics as a one-row table.
* `tables/degree_distribution.csv` - number of stations per degree value.
* `tables/component_sizes.csv` - size of every connected component.
* `tables/articulation_points.csv` - every cut vertex, with name / coordinates / region / degree.
* `tables/bridges.csv` - every cut edge, with endpoint names and trip frequency.
* `tables/stops_by_region.csv` - station counts per administrative region.
* `figures/degree_distribution.png`, `figures/components_summary_bar.png`, `figures/top_articulation_points.png`, `figures/network_overview_map.png`, `figures/stops_by_region.png`.

Nothing outside `outputs/nb/03_descriptive_analysis/` is touched.

## 1. Environment bootstrap

The cell below makes the notebook runnable both on a local checkout and on Google Colab. It defines `_ensure(...)`, which pip-installs only the packages that are genuinely missing (so re-running the notebook is cheap), and `find_repo_root()`, which walks up from the current directory looking for the GTFS folder and, failing that, clones the repository into `/content`. It then sets `REPO`, `DATA` and `OUT` and creates the notebook output root. Every later cell relies on these three paths, so this must run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders and tunable constants

We install and import the scientific stack (`pandas`, `numpy`, `networkx`, `matplotlib`, `seaborn`) and then fix the folder layout for this stage. Following the project convention, each notebook owns exactly one output folder: this one writes into `outputs/nb/03_descriptive_analysis/` (with `tables/` and `figures/` sub-folders) and reads the previous stage from `outputs/nb/02_graph_construction/`. The pre-existing `outputs/tables`, `outputs/figures` and `outputs/rail` folders hold the results cited in the written report and are deliberately never written to here.

The constants are gathered here so a grader can trade runtime for detail in one place. None of the algorithms in this notebook are expensive - connected components, articulation points and bridges are all linear-time DFS routines and finish in a few seconds on a 30k-node / 52k-edge graph - so the constants mainly control figure size and how many rows the top-N tables show. `FIG_DPI` is the only real cost knob: at 150 dpi the 30,000-point map takes a few seconds to rasterise.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn')

import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

PREV = OUT / '02_graph_construction'      # read-only: artifacts of notebook 02
STAGE = OUT / '03_descriptive_analysis'   # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
FIG_DPI = 150        # figure resolution; drop to 90 for faster, smaller files
TOP_N = 20           # rows shown in every top-N table / bar chart
MAP_ALPHA = 0.55     # point transparency on the ~30k-point geographic scatter
DEG_HIST_BINS = 60   # bins of the linear degree histogram

print('previous stage :', PREV)
print('this stage     :', STAGE)

## 3. Hebrew label rendering

Stop names in the Israeli GTFS feed are Hebrew, and two of the figures below (the top articulation points chart and the per-region chart) print them. Matplotlib does not implement the Unicode bidirectional algorithm, so right-to-left text comes out reversed and unreadable. The cell below monkey-patches `matplotlib.text.Text.set_text` once so that any string containing Hebrew characters is converted to display order via `python-bidi` before it is drawn, and selects a font that actually has Hebrew glyphs (Arial on Windows, DejaVu Sans everywhere else). It is idempotent - re-running it will not stack patches. All other text in the notebook is English, per the submission requirement.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Loading the graph produced by notebook 02

This stage depends on notebook `02_graph_construction`. Rather than un-pickling a `networkx` object (pickles are version-fragile and unreadable to a grader), we reload the two plain CSV tables that stage 02 exports and rebuild the graph objects from them - the CSVs contain exactly the same information. `find_artifact` searches the whole previous-stage folder so it works whether stage 02 put its tables at the folder root or inside `tables/`, and raises an explicit, actionable error if the artifacts are missing.

In [ ]:
# --- Locate the artifacts written by notebook 02 --------------------------
def find_artifact(stage_dir, filename):
    """Return the path of `filename` under a stage folder, or None if absent."""
    if not stage_dir.is_dir():
        return None
    direct = stage_dir / filename
    if direct.exists():
        return direct
    matches = sorted(stage_dir.rglob(filename))
    return matches[0] if matches else None

nodes_path = find_artifact(PREV, 'nodes.csv')
edges_path = find_artifact(PREV, 'edges.csv')
missing = [name for name, p in [('nodes.csv', nodes_path), ('edges.csv', edges_path)] if p is None]
if missing:
    raise FileNotFoundError(
        f"{', '.join(missing)} not found under {PREV} - "
        'run notebook 02_graph_construction first; it writes nodes.csv and edges.csv.'
    )

nodes_df = pd.read_csv(nodes_path, dtype={'stop_id': str}, encoding='utf-8-sig')
edges_df = pd.read_csv(edges_path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')
print(f'nodes: {len(nodes_df):,} rows  <-  {nodes_path}')
print(f'edges: {len(edges_df):,} rows  <-  {edges_path}')
nodes_df.head()

## 5. Rebuilding the directed and undirected graphs

The project's model is a **trip-adjacency graph**: a node is a stop that appears in at least one trip, and a directed edge `u -> v` exists when some trip visits `v` immediately after `u`; the edge weight is the number of trips that use that segment. `edges.csv` stores this directed graph. From it we derive the undirected projection `G` used for all connectivity work: one edge per unordered pair, with the weights of the two travel directions summed - this is what makes the undirected edge count (about 51.8k) slightly smaller than the directed one (about 52.0k), since most segments are served in both directions and collapse into a single edge.

Station attributes (name, coordinates, region, metropolitan area) are attached from `nodes.csv`. The helpers `_num` and `_txt` coerce blanks and `NaN` safely, so a missing coordinate becomes `None` rather than a silent `NaN` that would later be plotted at a nonsense position.

In [ ]:
# --- Rebuild the two graph objects ----------------------------------------
def _num(value):
    """Coerce to float; return None for blanks, NaN or non-numeric input."""
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


def build_graphs(nodes_df, edges_df):
    """Return (G, D): the undirected projection and the directed trip graph."""
    attr = {}
    for rec in nodes_df.to_dict('records'):
        attr[str(rec.get('stop_id'))] = {
            'stop_name': _txt(rec.get('stop_name')),
            'lat': _num(rec.get('lat')),
            'lon': _num(rec.get('lon')),
            'region': _txt(rec.get('region')),
            'metro': _txt(rec.get('metro')),
        }

    weight_col = next((c for c in ('trip_frequency', 'weight', 'count')
                       if c in edges_df.columns), None)

    D = nx.DiGraph()
    for rec in edges_df.to_dict('records'):
        u, v = str(rec['from_stop']), str(rec['to_stop'])
        D.add_edge(u, v, weight=int(rec[weight_col]) if weight_col else 1)

    # Undirected projection: one edge per unordered pair, weights summed.
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        if G.has_edge(u, v):
            G[u][v]['weight'] += data['weight']
        else:
            G.add_edge(u, v, weight=data['weight'])

    default = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}
    for graph in (G, D):
        for n in graph.nodes():
            graph.nodes[n].update(attr.get(n, default))
    return G, D


G, D = build_graphs(nodes_df, edges_df)
unnamed = sum(1 for n in G.nodes() if not G.nodes[n]['stop_name'])
print(f'undirected G : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'directed   D : {D.number_of_nodes():,} nodes, {D.number_of_edges():,} edges')
print(f'nodes with no name attribute: {unnamed:,}')

## 6. Global descriptive statistics

This is the core measurement cell. It computes, in one pass:

* **Size**: node and edge counts for both the undirected and the directed graph.
* **Density** `2m / (n(n-1))`: what fraction of all possible station pairs are directly connected. For a spatial transport network this is expected to be tiny.
* **Degree summary**: mean, median, min and max number of neighbouring stations.
* **Connected components** of `G`, sorted by size, plus the share of stations that live in the largest one - the standard measure of how fragmented a network already is before we attack it.
* **Weakly / strongly connected components** of `D`. The gap between them tells us how much of the network is served in both directions: a large strongly connected component means most station pairs have a return path.
* **Articulation points** (cut vertices) and **bridges** (cut edges). A cut vertex is a station whose removal increases the number of connected components; a bridge is a segment with the same property. `networkx` computes both with linear-time DFS algorithms (Hopcroft-Tarjan biconnected components, and chain decomposition for bridges), so this cell finishes in seconds rather than minutes. We wrap `articulation_points` in `sorted(set(...))` because older `networkx` releases could emit the same cut vertex more than once, which would inflate the count.

The result is written to `network_summary.json` and `tables/network_summary.csv` and displayed as a table.

In [ ]:
# --- Global descriptive statistics ----------------------------------------
components = sorted(nx.connected_components(G), key=len, reverse=True)
component_sizes = [len(c) for c in components]
largest = component_sizes[0]

# Cut vertices and cut edges: both are linear-time DFS routines.
ap = sorted(set(nx.articulation_points(G)))
br = list(nx.bridges(G))

wcc = list(nx.weakly_connected_components(D))
scc = list(nx.strongly_connected_components(D))

degrees = np.array([d for _, d in G.degree()])

summary = {
    'num_nodes': G.number_of_nodes(),
    'num_edges_undirected': G.number_of_edges(),
    'num_edges_directed': D.number_of_edges(),
    'density': round(nx.density(G), 6),
    'avg_degree': round(float(degrees.mean()), 2),
    'median_degree': float(np.median(degrees)),
    'min_degree': int(degrees.min()),
    'max_degree': int(degrees.max()),
    'num_connected_components': len(components),
    'largest_component_nodes': largest,
    'largest_component_share': round(largest / G.number_of_nodes(), 4),
    'num_weakly_connected': len(wcc),
    'num_strongly_connected': len(scc),
    'largest_scc_nodes': max(len(c) for c in scc),
    'num_articulation_points': len(ap),
    'num_bridges': len(br),
}

with open(STAGE / 'network_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
pd.DataFrame([summary]).to_csv(TABLES / 'network_summary.csv', index=False, encoding='utf-8-sig')

print('saved:', STAGE / 'network_summary.json')
pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})

## 7. Degree distribution

The degree of a station is the number of distinct stations directly reachable from it in one segment. Two views are drawn:

* **Top panel** - a linear histogram with a logarithmic y-axis. The log scale is a readability change over the original script: on a linear y-axis the bar for degree 2 is so tall that the entire tail is invisible.
* **Bottom panel** - the same data on log-log axes, the standard visual test for a heavy-tailed / power-law-like distribution. A straight line here is consistent with scale-free behaviour. We overlay an ordinary least-squares line and report its slope. This is a descriptive fit, **not** a rigorous power-law test (which would need maximum-likelihood estimation plus a Kolmogorov-Smirnov goodness-of-fit check); treat the slope as an indicator of tail heaviness, nothing more.

The per-degree counts are also saved as a table so the plot can be reproduced without re-running the notebook.

In [ ]:
# --- Degree distribution: histogram + log-log view -------------------------
deg_counts = pd.Series(degrees).value_counts().sort_index()
deg_counts = deg_counts[deg_counts.index > 0]

deg_table = pd.DataFrame({'degree': deg_counts.index.astype(int),
                          'num_stations': deg_counts.to_numpy().astype(int)})
deg_table['share'] = (deg_table['num_stations'] / len(degrees)).round(5)
deg_table.to_csv(TABLES / 'degree_distribution.csv', index=False, encoding='utf-8-sig')

x = np.log10(deg_table['degree'].to_numpy(dtype=float))
y = np.log10(deg_table['num_stations'].to_numpy(dtype=float))
slope, intercept = np.polyfit(x, y, 1)

fig, axes = plt.subplots(2, 1, figsize=(8, 10))

axes[0].hist(degrees, bins=DEG_HIST_BINS, color='#2563eb', edgecolor='white', linewidth=0.3)
axes[0].set_yscale('log')
axes[0].set_xlabel('Degree (number of neighbouring stations)')
axes[0].set_ylabel('Number of stations (log scale)')
axes[0].set_title('Degree distribution - Israel public transport network')

axes[1].scatter(x, y, s=12, color='#dc2626', alpha=0.65, label='observed')
axes[1].plot(x, slope * x + intercept, color='#111827', linewidth=1.2, linestyle='--',
             label=f'least-squares fit, slope = {slope:.2f}')
axes[1].set_xlabel('log10(degree)')
axes[1].set_ylabel('log10(number of stations)')
axes[1].set_title('Log-log degree distribution (heavy-tail check)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'degree_distribution.png', dpi=FIG_DPI)
plt.show()

print(f'degree range: {degrees.min()} to {degrees.max()}, mean {degrees.mean():.2f}, median {np.median(degrees):.0f}')
print(f'stations with degree <= 2: {(degrees <= 2).sum():,} ({(degrees <= 2).mean():.1%})')
print(f'log-log least-squares slope: {slope:.2f}')

## 8. Connected components

A connected component is a maximal set of stations that can reach each other through some sequence of segments. If the network were one component, every stop in the country would be reachable from every other. It is not - a handful of small isolated clusters exist (typically local services that never touch the national network in this feed snapshot). The bar chart shows the largest components on a logarithmic y-axis, because the gap between the giant component and the rest spans four orders of magnitude and would otherwise be unreadable. The full list of component sizes is saved to `tables/component_sizes.csv`.

In [ ]:
# --- Connected component sizes --------------------------------------------
comp_table = pd.DataFrame({'component_rank': range(1, len(component_sizes) + 1),
                           'num_stations': component_sizes})
comp_table['share'] = (comp_table['num_stations'] / G.number_of_nodes()).round(6)
comp_table.to_csv(TABLES / 'component_sizes.csv', index=False, encoding='utf-8-sig')

top_components = component_sizes[:TOP_N]
bar_colors = ['#2563eb' if i == 0 else '#94a3b8' for i in range(len(top_components))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, len(top_components) + 1), top_components, color=bar_colors)
ax.set_yscale('log')
ax.set_xticks(range(1, len(top_components) + 1))
ax.set_xlabel('Connected component (rank by size)')
ax.set_ylabel('Number of stations (log scale)')
ax.set_title(f'Connected component sizes - top {len(top_components)} of {len(component_sizes)}')
plt.tight_layout()
plt.savefig(FIGURES / 'components_summary_bar.png', dpi=FIG_DPI)
plt.show()

print(f'{len(component_sizes)} components; largest holds '
      f"{largest:,} stations ({largest / G.number_of_nodes():.2%})")
comp_table.head(TOP_N)

## 9. Articulation points and bridges

These are the structural single points of failure and the reason this notebook matters for the project's resilience question.

* An **articulation point** (cut vertex) is a station whose removal splits its component into two or more pieces. Passengers on one side lose all connectivity to the other side, not just a slower route.
* A **bridge** (cut edge) is a segment with the same property - the only physical link between two parts of the network.

Both tables are exported in full, enriched with stop name, coordinates, region and degree (for articulation points) or endpoint names and trip frequency (for bridges), so that later notebooks and the written report can cross-reference them. Sorting bridges by trip frequency immediately surfaces the single-link segments that carry the most service - the highest-impact cut edges.

In [ ]:
# --- Export cut vertices and cut edges -------------------------------------
ap_df = pd.DataFrame([{
    'stop_id': n,
    'stop_name': G.nodes[n]['stop_name'],
    'lat': G.nodes[n]['lat'],
    'lon': G.nodes[n]['lon'],
    'region': G.nodes[n]['region'],
    'metro': G.nodes[n]['metro'],
    'degree': G.degree(n),
} for n in ap])
ap_df = ap_df.sort_values('degree', ascending=False).reset_index(drop=True)
ap_df.to_csv(TABLES / 'articulation_points.csv', index=False, encoding='utf-8-sig')

br_df = pd.DataFrame([{
    'from_stop': u,
    'to_stop': v,
    'from_name': G.nodes[u]['stop_name'],
    'to_name': G.nodes[v]['stop_name'],
    'trip_frequency': int(G[u][v].get('weight', 1)),
} for u, v in br])
br_df = br_df.sort_values('trip_frequency', ascending=False).reset_index(drop=True)
br_df.to_csv(TABLES / 'bridges.csv', index=False, encoding='utf-8-sig')

print(f'articulation points : {len(ap_df):,} '
      f"({len(ap_df) / G.number_of_nodes():.2%} of all stations)")
print(f'bridges             : {len(br_df):,} '
      f"({len(br_df) / G.number_of_edges():.2%} of all undirected edges)")
print('\nHighest-traffic bridges:')
display(br_df.head(TOP_N))
print('Highest-degree articulation points:')
ap_df.head(TOP_N)

## 10. Which stations are the biggest cut vertices?

Not all articulation points are equally important. A cut vertex of degree 2 separates a dead-end branch of a few stops; a cut vertex of degree 30 is a hub whose loss would strand a whole sub-network. The horizontal bar chart below ranks the top articulation points by degree, using the real Hebrew stop names (rendered correctly thanks to the bidi patch installed in section 3). This is the shortlist that the later resilience notebooks should focus on.

In [ ]:
# --- Top articulation points by degree -------------------------------------
top_ap = ap_df.head(TOP_N).iloc[::-1]   # reversed so the largest ends up on top
labels = [f'{name} ({sid})' for name, sid in zip(top_ap['stop_name'], top_ap['stop_id'])]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(range(len(top_ap)), top_ap['degree'].to_numpy(), color='#7c3aed')
ax.set_yticks(range(len(top_ap)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Degree (number of neighbouring stations)')
ax.set_title(f'Top {len(top_ap)} articulation points by degree')
plt.tight_layout()
plt.savefig(FIGURES / 'top_articulation_points.png', dpi=FIG_DPI)
plt.show()

## 11. Geographic overview of the network

A scatter of every station at its true longitude / latitude, coloured and sized by degree, gives an immediate sense of where the network is dense (the coastal Tel Aviv - Haifa strip and Jerusalem) and where it thins out (the Negev). This is a plain lat/lon scatter, not a projected map, so we set the aspect ratio to `1 / cos(mean latitude)` to keep the country's shape approximately correct instead of horizontally stretched.

The coordinate filter here is a deliberate fix to the original script, which used `if d.get("lat") and d.get("lon")`. That test is falsy for a coordinate of exactly `0.0` and, worse, is truthy for `NaN` - so missing coordinates would have been plotted. We test explicitly for a present, finite value instead.

In [ ]:
# --- Network overview map (lat/lon scatter) --------------------------------
def has_coords(data):
    """True only when both coordinates are present and finite (0.0 included)."""
    lat, lon = data.get('lat'), data.get('lon')
    return lat is not None and lon is not None and np.isfinite(lat) and np.isfinite(lon)

coord_nodes = [(n, d) for n, d in G.nodes(data=True) if has_coords(d)]
if not coord_nodes:
    raise ValueError('No station carries usable coordinates - check nodes.csv from notebook 02.')

lats = [d['lat'] for _, d in coord_nodes]
lons = [d['lon'] for _, d in coord_nodes]
degs = [G.degree(n) for n, _ in coord_nodes]
max_deg = max(degs)
sizes = [2 + 30 * (d / max_deg) for d in degs]

fig, ax = plt.subplots(figsize=(8, 11))
sc = ax.scatter(lons, lats, c=degs, s=sizes, cmap='viridis', alpha=MAP_ALPHA, linewidths=0)
plt.colorbar(sc, ax=ax, label='Degree')
ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(lats)))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Israel public transport stations\n'
             f'({len(coord_nodes):,} active stops with coordinates)')
plt.tight_layout()
plt.savefig(FIGURES / 'network_overview_map.png', dpi=FIG_DPI)
plt.show()

print(f'stations with coordinates: {len(coord_nodes):,} of {G.number_of_nodes():,}')

## 12. Stations per region

Finally, a simple count of active stations per administrative region, which puts the geography above into numbers and is a useful sanity check that the region attribute survived the join in notebook 02. The bar colours are generated from a seaborn palette sized to the number of regions actually present - the original script hard-coded four colours, which would have silently recycled or run out if the region set ever changed. Region names are Hebrew and are rendered through the bidi patch.

In [ ]:
# --- Stations per region ---------------------------------------------------
region_counts = (pd.Series([G.nodes[n]['region'] or 'Unknown' for n in G.nodes()])
                 .value_counts())
region_table = region_counts.rename_axis('region').reset_index(name='num_stations')
region_table['share'] = (region_table['num_stations'] / G.number_of_nodes()).round(4)
region_table.to_csv(TABLES / 'stops_by_region.csv', index=False, encoding='utf-8-sig')

palette = sns.color_palette('deep', len(region_counts))
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(r) for r in region_counts.index], region_counts.to_numpy(), color=palette)
ax.set_xlabel('Region')
ax.set_ylabel('Number of stations')
ax.set_title('Active stations by administrative region')
for bar, val in zip(bars, region_counts.to_numpy()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:,}', ha='center', va='bottom', fontsize=10)
ax.margins(y=0.12)
plt.tight_layout()
plt.savefig(FIGURES / 'stops_by_region.png', dpi=FIG_DPI)
plt.show()

region_table

## Takeaways

* **The network is large and extremely sparse.** About 30,463 active stations and 51,772 undirected segments give a density near 0.0001 and an average degree of roughly 3.4 - a typical station touches only three or four others. That is normal for a spatial transport network: geography, not popularity, limits who can be a neighbour.
* **It is effectively one network.** Around 12 connected components exist, but the largest holds about 99.3% of all stations; the rest are tiny isolated clusters. Any resilience statement in later notebooks therefore concerns the giant component, and "the network fragmented" has to mean something stronger than "a few stops fell off".
* **The degree distribution is heavy-tailed, but do not over-claim.** Most stations sit at degree 2 (a stop on a line), while a small number of interchanges reach several dozen neighbours. The log-log plot is roughly linear over the bulk of the range, which is *consistent with* scale-free-like behaviour - but the least-squares slope printed above is a descriptive fit, not a validated power-law exponent, and the tail is short. We report it as "heavy-tailed", not "scale-free".
* **Single points of failure are real but a small minority.** Roughly 900 articulation points (about 3% of stations) and 971 bridges (under 2% of edges) are structural single points of failure. Two honest caveats: many of these are low-degree dead-end branches whose loss disconnects only a handful of stops, which is why section 10 ranks them by degree; and this is a purely topological notion of failure - it ignores how many passengers actually use a segment, which is exactly what the centrality and resilience notebooks add next.
* **A largely bidirectional network.** The directed graph has only slightly more edges than the undirected projection, and the largest strongly connected component covers most of the network, meaning that for the vast majority of station pairs a return path exists. Segments served in one direction only are the exception.
* **Geography dominates.** Stations concentrate heavily in the Centre district and the coastal metropolitan strip; the southern periphery is sparse. This is the structural background against which the socio-economic findings in the later notebooks should be read.